# CNN t1 quantizada (8 bits) no SAT-6 

Notebook enxuto contendo apenas o necessário para:
1. Carregar o dataset SAT-6 (já "pickled" pelo `0_pickle_dataset.ipynb`);
2. Definir e treinar a topologia **t1** com quantização de **8 bits** (pesos e ativações);
3. Testar/avaliar a acurácia do modelo;
4. Exportar o modelo para **QONNX**, formato de entrada aceito pelo **hls4ml** (ao contrário do FINN, não é necessário aplicar as transformações de streamlining/conversão para hardware — isso é feito pelo próprio hls4ml a partir do ONNX/QONNX).

In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from brevitas.export import export_qonnx
from utils.utils import read_object
from cnn_base import t1_CNN

%matplotlib inline
torch.manual_seed(4)

## 1. Dataset

A classe `SatImgDataset` precisa estar definida antes de desserializar os `.pkl` (mesmo nome/estrutura usada em `0_pickle_dataset.ipynb`).

In [2]:
batch_size = 512
epochs = 30
lr = 3e-4
device = torch.device('cpu')

train_dataset_output_name = "./dataset/train_dataset.pkl"
test_dataset_output_name = "./dataset/test_dataset.pkl"

dataset_train = read_object(train_dataset_output_name)
dataset_test = read_object(test_dataset_output_name)

loader_train = DataLoader(dataset_train, batch_size, shuffle=True)
loader_test = DataLoader(dataset_test, batch_size, shuffle=False)

## 2. Treinamento

In [3]:
def train_model(model, loader_train, optimizer, criterion, epochs, device):
    for e in range(epochs):
        model.train()
        running_loss = 0.0
        print(f"epoch ({e})")
        for x,y in tqdm(loader_train):
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long)

            pred = model(x)
            loss = criterion(pred, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        mean_loss = running_loss / len(loader_train)
        print(f"epoch ({e}) loss: {mean_loss:.4f}")

    return model


model = t1_CNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

model_output_filename = "./pytorch_models/sat6-cnn-t1.pt"
model = train_model(model, loader_train, optimizer, criterion, epochs, device)

os.makedirs("./pytorch_models", exist_ok=True)
torch.save(model.state_dict(), model_output_filename)
print(f"modelo salvo em {model_output_filename}")

epoch (0)


100%|██████████| 633/633 [00:09<00:00, 65.24it/s]


epoch (0) loss: 0.6262
epoch (1)


100%|██████████| 633/633 [00:09<00:00, 63.58it/s]


epoch (1) loss: 0.2021
epoch (2)


100%|██████████| 633/633 [00:09<00:00, 65.28it/s]


... (51 outputs left out)

100%|██████████| 633/633 [00:09<00:00, 64.67it/s]


epoch (28) loss: 0.0611
epoch (29)


100%|██████████| 633/633 [00:10<00:00, 61.98it/s]

epoch (29) loss: 0.0587
modelo salvo em ./pytorch_models/sat6-cnn-t1.pt


## 4. Teste / avaliação

In [4]:
def test_model(model, loader_test, dataset_test, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x,y in loader_test:
            x = x.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.long)

            pred = model(x)
            predicted = torch.max(pred, 1)[1]
            correct += (predicted == y).sum()
    accuracy = correct.item() / len(dataset_test) * 100
    print(f"acurácia do modelo t1: {accuracy:.4f}% ({len(dataset_test)} casos de teste)")
    return accuracy


test_model(model, loader_test, dataset_test, device)

acurácia do modelo t1: 97.9407% (81000 casos de teste)


97.94074074074074

## 5. Exportação para QONNX (entrada do hls4ml)

Diferente do fluxo com o FINN (que exige as transformações de *streamlining*, *lowering* e conversão para camadas de hardware antes da síntese), o **hls4ml** consegue consumir o modelo diretamente a partir do QONNX exportado pelo Brevitas, aplicando internamente o parsing das camadas quantizadas (`Quant`, `QuantConv`, `QuantLinear`, etc.). Portanto, basta exportar e limpar o grafo ONNX.

In [5]:
import qonnx.util.cleanup
import onnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.gemm_to_matmul import GemmToMatMul
from qonnx.util.to_channels_last import to_channels_last


os.makedirs("./hls4ml_onnxs", exist_ok=True)
export_onnx_path = "./hls4ml_onnxs/sat6-cnn-t1.onnx"

model.eval()
export_qonnx(model,  torch.randn(1, 4, 28, 28), export_path=export_onnx_path, opset_version=14)
print(f"modelo QONNX exportado em {export_onnx_path}")

# Clean and convert Gemm → MatMul
qonnx.util.cleanup.cleanup(export_onnx_path, out_file=export_onnx_path)
model_wrapper = ModelWrapper(export_onnx_path)
model_wrapper = model_wrapper.transform(GemmToMatMul())
model_wrapper = qonnx.util.cleanup.cleanup_model(model_wrapper)
model_wrapper.save(export_onnx_path)
print(f'Cleaned QONNX saved to {export_onnx_path}')


modelo QONNX exportado em ./hls4ml_onnxs/sat6-cnn-t1.onnx
Cleaned QONNX saved to ./hls4ml_onnxs/sat6-cnn-t1.onnx


/home/fauri/Documentos/hls4ml/.venv_hls4ml_p/lib/python3.10/site-packages/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


## Transform in Channel last if it's needed for hls4ml

In [6]:
to_channels_last(
    in_file=export_onnx_path,
    out_file=export_onnx_path,
    make_input_channels_last=True
)
print(f'Model layout converted to channels_last and saved to {export_onnx_path}')

Model layout converted to channels_last and saved to ./hls4ml_onnxs/sat6-cnn-t1.onnx


### Verify if every node has name

If not, create a name for this nodes and save the new onnx model.

In [7]:
onnx_model = onnx.load(export_onnx_path)

def onnx_node_names(onnx_model):
    graph = onnx_model.graph
    for i, node in enumerate(graph.node):
        print(f"Node {i}: {node.name}")
        if not node.name:
            node.name = f"{node.op_type.lower()}_{i}"
            print(f"Node {i} had no name. Assigned name: {node.name}")
            
    for i, value_info in enumerate(graph.value_info):
        print(f"ValueInfo {i}: {value_info.name}")
        if not value_info.name:
            value_info.name = f"tensor_{i}"
            print(f"ValueInfo {i} had no name. Assigned name: {value_info.name}")

onnx_node_names(onnx_model)   

Node 0: Conv_0
Node 1: Relu_0
Node 2: Conv_1
Node 3: Relu_1
Node 4: 
Node 4 had no name. Assigned name: flatten_4
Node 5: MatMul_0
Node 6: Add_0
ValueInfo 0: Conv_0_param1
ValueInfo 1: Add_0_param0
ValueInfo 2: Conv_1_param1
ValueInfo 3: MatMul_0_out0
ValueInfo 4: 07DrIQ
ValueInfo 5: LWsjup
ValueInfo 6: 1lBoVC
ValueInfo 7: Conv_1_out0
ValueInfo 8: pTT7AZ
ValueInfo 9: qVPd3Y
ValueInfo 10: MatMul_0_param0
ValueInfo 11: twsLBD


In [8]:
# Caso precisar, mas melhor não usar.
def fix_onnx_node_names(model_path, output_path):
    model = onnx.load(model_path)
    graph = model.graph
    
    # 1. Corrige os nós do grafo
    for i, node in enumerate(graph.node):
        if not node.name:  # Se o nome estiver vazio ("")
            # Cria um nome baseado no tipo do operador e no índice
            node.name = f"{node.op_type.lower()}_{i}"
            
    # 2. Garante que os tensores intermediários (valores) também tenham nome se necessário
    for i, value_info in enumerate(graph.value_info):
        if not value_info.name:
            value_info.name = f"tensor_{i}"
            
    onnx.save(model, output_path)
    print(f"Modelo corrigido salvo em: {output_path}")


fix_onnx_node_names(export_onnx_path, export_onnx_path)

Modelo corrigido salvo em: ./hls4ml_onnxs/sat6-cnn-t1.onnx
